In [ ]:
import copy

# --- 1. データ構造の定義 ---

class ASTNode:
    """数式の一部を表現する最小単位（簡易版）"""
    def __init__(self, value, left=None, right=None, type="op"):
        self.value = value  # '+', '-', '*', '/', '==', または 'x', '6' など
        self.left = left
        self.right = right
        self.type = type # "op", "term", "equation", "logical"

    def __str__(self):
        if self.left and self.right:
            return f"({self.left} {self.value} {self.right})"
        return str(self.value)

class Context:
    """前提条件や定数を保持"""
    def __init__(self, assumptions=None):
        self.assumptions = assumptions or []
    
    def __repr__(self):
        return f"Context({self.assumptions})"

class Proposition:
    """エッジを流れる「命題」。式と文脈のペア。"""
    def __init__(self, ast, context=None):
        self.ast = ast
        self.context = context or Context()

    def __repr__(self):
        return f"Prop: {self.ast} | {self.context}"

# --- 2. 関数ライブラリ (MIS Nodes) ---

def MapGlobal(prop, op, operand):
    """
    等式の両辺に同じ操作を適用する (A = B -> A/2 = B/2)
    """
    print(f"--- Node: MapGlobal [{op} {operand}] ---")
    new_ast = copy.deepcopy(prop.ast)
    # 両辺を新しい演算ノードで包む（簡易実装）
    new_ast.left = ASTNode(op, new_ast.left, ASTNode(operand, type="term"))
    new_ast.right = ASTNode(op, new_ast.right, ASTNode(operand, type="term"))
    return Proposition(new_ast, prop.context)

def RewriteLocal(prop, target_side, rule_name, new_val):
    """
    式の特定部位を書き換える (x^2-4x+3 -> (x-1)(x-3))
    ※ 実際はASTを解析して自動計算するが、ここでは明示的に書き換え。
    """
    print(f"--- Node: RewriteLocal [{rule_name} on {target_side}] ---")
    new_ast = copy.deepcopy(prop.ast)
    if target_side == "LHS":
        new_ast.left = ASTNode(new_val, type="term")
    else:
        new_ast.right = ASTNode(new_val, type="term")
    return Proposition(new_ast, prop.context)

def SplitContext(prop, var, cases):
    """
    世界を分岐させる。
    """
    print(f"--- Node: SplitContext [on {var}] ---")
    branches = {}
    for case in cases:
        new_context = copy.deepcopy(prop.context)
        new_context.assumptions.append(f"{var} {case}")
        branches[case] = Proposition(copy.deepcopy(prop.ast), new_context)
    return branches

# --- 3. 実行デモ (2x^2 - 8x + 6 = 0 を解く) ---

# 初期状態: 2x^2 - 8x + 6 = 0
start_ast = ASTNode("==", 
                    left=ASTNode("2x^2 - 8x + 6", type="term"), 
                    right=ASTNode("0", type="term"))
p0 = Proposition(start_ast)
print(f"Step 0 (Start): {p0}")

# Step 1: 両辺を2で割る
p1 = MapGlobal(p0, "/", 2)
print(f"Step 1: {p1}")

# Step 2: 左辺を因数分解 (Local Rewrite)
p2 = RewriteLocal(p1, "LHS", "Factorization", "(x-1)(x-3)")
print(f"Step 2: {p2}")

# Step 3: 文字係数だった場合の分岐シミュレーション (a!=0 の仮定など)
branches = SplitContext(p2, "a", ["!= 0", "== 0"])
for c, p in branches.items():
    print(f"Branch [{c}]: {p}")